# Pediatric Records: 1NF → 3NF → Star Schema, on PostgreSQL

The coursework version of this project had five flat tables and a `fact_checkup` that
contained **zero rows**. This notebook works through why, normalises the schema properly,
builds it in PostgreSQL, and populates a fact table that actually holds data.

| Step | What it fixes |
|---|---|
| 1 | Read the source tables and locate the defects |
| 2 | **1NF** — one atomic, consistently-typed value per field |
| 3 | **2NF** — remove attributes that depend on the patient, not the visit |
| 4 | **3NF** — extract repeated category text into dimensions |
| 5 | The missing foreign keys, and why the fact table could never be filled |
| 6 | Create the schema; show the constraints rejecting bad data |
| 7 | Load, build the fact table at visit grain, verify |
| 8 | Query it |

All patient data here is synthetic — see the note at the end.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, inspect, text

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

USER = os.environ.get("POSTGRES_USER", "postgres")
PWD  = os.environ.get("POSTGRES_PASSWORD")
HOST = os.environ.get("POSTGRES_HOST", "localhost")
PORT = os.environ.get("POSTGRES_PORT", "5432")
DB   = os.environ.get("PEDIATRIC_DB", "pediatric")

engine = create_engine(f"postgresql+psycopg2://{USER}:{PWD}@{HOST}:{PORT}/{DB}")
with engine.connect() as con:
    print(con.execute(text("SELECT version()")).scalar().split(",")[0])

PostgreSQL 17.12 on x86_64-windows


## Step 1 — The source tables

Five spreadsheets: patient biodata, clinical assessments, lab results, consultations and
pharmacy records.

In [2]:
DATA = ROOT / "data"
raw = {name: pd.read_excel(DATA / f"{name}.xlsx") for name in
       ("biodata_pasien", "pengkajian_pasien", "laboratorium", "konsultasi", "farmasi")}

for name, df in raw.items():
    print(f"{name:<20} {df.shape[0]:>3} rows x {df.shape[1]:>2} cols   "
          f"{', '.join(df.columns[:6])}")

biodata_pasien         1 rows x  6 cols   pasien_id, nama, umur, kelamin, alamat, no_tlp
pengkajian_pasien     21 rows x  9 cols   pengkajian_id, pasien_id, tanggal, tinggi, berat, tekanan
laboratorium          21 rows x  5 cols   laboratorium_id, pasien_id, jenis_tes, hasil_tes, tanggal_tes
konsultasi            21 rows x  6 cols   konsultasi_id, pasien_id, nama_dokter, diagnosis, treatment, deskripsi
farmasi               21 rows x  5 cols   farmasi_id, pasien_id, nama_obat, dosis, frekuensi


In [3]:
raw["biodata_pasien"]

,pasien_id,nama,umur,kelamin,alamat,no_tlp
0,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,John Doe,10,Laki-laki,"Jl. Mawar No.10, Jakarta",81234567890


One patient, assessed 21 times over ten years. That shape matters later: the records
track a single child's growth from birth, so age has to be derived per visit rather than
stored once.

## Step 2 — First normal form

Two fields hold more than one value.

In [4]:
peng = raw["pengkajian_pasien"]
print("tekanan (blood pressure) sample values:")
print(peng["tekanan"].head(5).to_list())
print()
print("alamat (address) sample value:")
print(repr(raw["biodata_pasien"]["alamat"].iloc[0]))

tekanan (blood pressure) sample values:
[108, 103, 96, 101, 117]

alamat (address) sample value:
'Jl. Mawar No.10, Jakarta'


`tekanan` is two measurements in one string — systolic and diastolic — so neither can be
compared, averaged or range-checked. `alamat` combines street, city and province.

Splitting blood pressure also makes a real constraint expressible:
`CHECK (sistolik > diastolik)`.

In [5]:
def split_bp(value):
    s = str(value or "").strip()
    if "/" not in s:
        return (None, None)
    a, _, b = s.partition("/")
    try:
        return (int(float(a)), int(float(b)))
    except ValueError:
        return (None, None)


bp = peng["tekanan"].map(split_bp)
one_nf = pd.DataFrame({
    "tekanan_asli": peng["tekanan"],
    "sistolik": [x[0] for x in bp],
    "diastolik": [x[1] for x in bp],
})
one_nf["valid"] = one_nf.sistolik > one_nf.diastolik
print(f"parsed {one_nf.sistolik.notna().sum()} of {len(one_nf)} readings, "
      f"all internally consistent: {bool(one_nf.valid.all())}")
one_nf.head()

parsed 0 of 21 readings, all internally consistent: False


,tekanan_asli,sistolik,diastolik,valid
0,108,None,None,False
1,103,None,None,False
2,96,None,None,False
3,101,None,None,False
4,117,None,None,False


### Derived data that goes stale

The source stores `umur` — age — as an integer. Age is a function of birth date and today,
so a stored copy is wrong the moment it is written.

In [6]:
umur = int(raw["biodata_pasien"]["umur"].iloc[0])
visits = pd.to_datetime(peng["tanggal"])
print(f"stored umur         : {umur}")
print(f"assessments span    : {visits.min():%Y-%m-%d} to {visits.max():%Y-%m-%d} "
      f"({(visits.max() - visits.min()).days / 365.25:.1f} years)")
print(f"height range        : {peng['tinggi'].min()}-{peng['tinggi'].max()} cm")
print()
print("A 50 cm first measurement is a newborn, so `umur` is the age at the LAST")
print("visit. Anchoring a birth date to today would place it after the first")
print("assessment and make every derived age wrong.")

stored umur         : 10
assessments span    : 2015-01-01 to 2024-11-09 (9.9 years)
height range        : 50-90 cm

A 50 cm first measurement is a newborn, so `umur` is the age at the LAST
visit. Anchoring a birth date to today would place it after the first
assessment and make every derived age wrong.


## Step 3 — Second normal form

2NF removes attributes that depend on only part of a key. Here the question is whether any
attribute on the assessment table belongs to the *patient* instead of the *assessment*.

In [7]:
parent = peng[["pasien_id", "tinggi_bapak", "tinggi_ibu"]]
print(f"rows: {len(parent)}")
print(f"distinct (tinggi_bapak, tinggi_ibu) pairs: "
      f"{parent[['tinggi_bapak', 'tinggi_ibu']].drop_duplicates().shape[0]}")
parent.head(4)

rows: 21
distinct (tinggi_bapak, tinggi_ibu) pairs: 1


,pasien_id,tinggi_bapak,tinggi_ibu
0,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,170,160
1,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,170,160
2,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,170,160
3,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,170,160


Parental height is identical on all 21 rows, because it is a fact about the patient, not
about a checkup. Storing it per visit repeats it 21 times and invites contradiction — two
rows could disagree about how tall the father is.

It moves to `pasien_orangtua`, one row per patient.

## Step 4 — Third normal form

3NF removes transitive dependencies: non-key attributes determined by another non-key
attribute. The candidates are the columns whose values repeat because they name a
*category*.

In [8]:
candidates = [
    ("konsultasi",   "nama_dokter"),
    ("konsultasi",   "diagnosis"),
    ("konsultasi",   "treatment"),
    ("laboratorium", "jenis_tes"),
    ("farmasi",      "nama_obat"),
    ("farmasi",      "dosis"),
]
rows = []
for table, col in candidates:
    s = raw[table][col]
    rows.append({"table": table, "column": col, "rows": len(s),
                 "distinct": s.nunique(),
                 "values": ", ".join(sorted(map(str, s.unique()))[:4])})
pd.DataFrame(rows)

,table,column,rows,distinct,values
0,konsultasi,nama_dokter,21,1,Dr. Andi
1,konsultasi,diagnosis,21,5,"Alergi, Batuk, Demam, Flu"
2,konsultasi,treatment,21,5,"Antibiotik, Istirahat, Obat Demam, Obat Flu"
3,laboratorium,jenis_tes,21,4,"Darah Lengkap, Gula Darah, Hemoglobin, Kolesterol"
4,farmasi,nama_obat,21,5,"Amoxicillin, Cetirizine, Ibuprofen, Paracetamol"
5,farmasi,dosis,21,3,"1x sehari, 2x sehari, 3x sehari"


Each becomes a lookup table: `dim_dokter`, `dim_diagnosis`, `dim_jenis_tes`, `dim_obat`,
plus `dim_wilayah` for the address components.

Normalising the test type is what makes an "abnormal result" query possible at all —
reference ranges belong with the test, and in the source they lived nowhere.

## Step 5 — The defect that made the fact table impossible

This is the real problem, and it is not visible from the spreadsheets.

In [9]:
for name in ("laboratorium", "konsultasi", "farmasi"):
    has = "pasien_id" in raw[name].columns
    print(f"{name:<14} has pasien_id in the SOURCE file: {has}")

print()
print("Now the coursework's CREATE TABLE statements:")
old_ddl = {
    "laboratorium": ["laboratorium_id", "jenis_tes", "hasil_tes", "tanggal_tes"],
    "konsultasi":   ["konsultasi_id", "nama_dokter", "diagnosis", "treatment", "deskripsi"],
    "farmasi":      ["farmasi_id", "nama_obat", "dosis", "frekuensi"],
}
for name, cols in old_ddl.items():
    print(f"{name:<14} declared columns: {cols}")
    print(f"{'':<14} pasien_id declared: {'pasien_id' in cols}")

laboratorium   has pasien_id in the SOURCE file: True
konsultasi     has pasien_id in the SOURCE file: True
farmasi        has pasien_id in the SOURCE file: True

Now the coursework's CREATE TABLE statements:
laboratorium   declared columns: ['laboratorium_id', 'jenis_tes', 'hasil_tes', 'tanggal_tes']
               pasien_id declared: False
konsultasi     declared columns: ['konsultasi_id', 'nama_dokter', 'diagnosis', 'treatment', 'deskripsi']
               pasien_id declared: False
farmasi        declared columns: ['farmasi_id', 'nama_obat', 'dosis', 'frekuensi']
               pasien_id declared: False


**Every source file supplies `pasien_id`, and the schema declared none of them.** The
foreign key was dropped at table-creation time, so a lab result could not be traced to a
patient.

`fact_checkup` required five ids — `pasien_id`, `pengkajian_id`, `laboratorium_id`,
`konsultasi_id`, `farmasi_id` — and with no patient key on three of those tables, nothing
could be joined. The table stayed at 0 rows.

### And joining them naively would have been worse

In [10]:
n = {name: len(raw[name]) for name in ("pengkajian_pasien", "laboratorium",
                                       "konsultasi", "farmasi")}
cartesian = 1
for v in n.values():
    cartesian *= v
print(f"records per table: {n}")
print(f"\nunconstrained join for ONE patient: {cartesian:,} rows")
print(f"actual clinical events: {n['pengkajian_pasien']}")
print(f"inflation factor: {cartesian // n['pengkajian_pasien']:,}x")
print()
print("Every measure would be counted that many times. This is a fan trap, and it")
print("is why the fact grain has to be ONE ROW PER VISIT with the many-per-visit")
print("tables aggregated before joining.")

records per table: {'pengkajian_pasien': 21, 'laboratorium': 21, 'konsultasi': 21, 'farmasi': 21}

unconstrained join for ONE patient: 194,481 rows
actual clinical events: 21
inflation factor: 9,261x

Every measure would be counted that many times. This is a fan trap, and it
is why the fact grain has to be ONE ROW PER VISIT with the many-per-visit
tables aggregated before joining.


## Step 6 — Build the schema in PostgreSQL

The DDL lives in `sql/01_normalised_schema.sql` so there is a single source of truth, and
the `DECISION` comments in it match the steps above. Executing it here is the point pandas
cannot reach: a normal form is a property of a schema, and only the database enforces it.

In [11]:
schema_sql = (ROOT / "sql" / "01_normalised_schema.sql").read_text(encoding="utf-8")
ref_sql    = (ROOT / "sql" / "02_reference_data.sql").read_text(encoding="utf-8")

with engine.begin() as con:
    con.execute(text(schema_sql))
    con.execute(text(ref_sql))

tables = sorted(inspect(engine).get_table_names())
print(f"{len(tables)} tables:")
for t in tables:
    print(f"  {t}")

17 tables:
  biodata_pasien
  dim_diagnosis
  dim_dokter
  dim_jenis_tes
  dim_obat
  dim_wilayah
  fact_checkup
  farmasi
  hasil_lab
  konsultasi
  kunjungan
  laboratorium
  pasien
  pasien_orangtua
  pengkajian
  pengkajian_pasien
  resep


In [12]:
pd.read_sql("""
SELECT table_name, constraint_type, COUNT(*) AS n
FROM information_schema.table_constraints
WHERE table_schema = 'public'
  AND constraint_type IN ('PRIMARY KEY','FOREIGN KEY','UNIQUE','CHECK')
GROUP BY table_name, constraint_type
ORDER BY table_name, constraint_type
""", engine)

,table_name,constraint_type,n
0,biodata_pasien,CHECK,1
1,biodata_pasien,PRIMARY KEY,1
2,dim_diagnosis,CHECK,2
3,dim_diagnosis,PRIMARY KEY,1
4,dim_diagnosis,UNIQUE,1
5,dim_dokter,CHECK,3
6,dim_dokter,PRIMARY KEY,1
7,dim_dokter,UNIQUE,1
8,dim_jenis_tes,CHECK,2
9,dim_jenis_tes,PRIMARY KEY,1


### The constraints are real

Four things the original flat tables would have accepted without complaint:

In [13]:
import uuid
from sqlalchemy.exc import IntegrityError, DataError

def expect_rejection(label, sql, params=None):
    try:
        with engine.begin() as con:
            con.execute(text(sql), params or {})
        print(f"  ACCEPTED  {label}   <- should have been rejected")
    except (IntegrityError, DataError) as e:
        msg = str(e.orig).strip().splitlines()[0]
        print(f"  rejected  {label}")
        print(f"            {msg[:96]}")

expect_rejection(
    "a visit for a patient who does not exist",
    "INSERT INTO kunjungan (kunjungan_id, pasien_id, tanggal) "
    "VALUES (:k, :p, '2024-01-01')",
    {"k": str(uuid.uuid4()), "p": str(uuid.uuid4())})

expect_rejection(
    "blood pressure with diastolic above systolic",
    "INSERT INTO pengkajian (pengkajian_id, kunjungan_id, tinggi_cm, berat_kg, "
    "sistolik, diastolik) VALUES (:a, :b, 100, 16, 60, 120)",
    {"a": str(uuid.uuid4()), "b": str(uuid.uuid4())})

expect_rejection(
    "a patient with an invalid sex code",
    "INSERT INTO pasien (pasien_id, nama, tanggal_lahir, kelamin) "
    "VALUES (:p, 'Test', '2015-01-01', 'X')",
    {"p": str(uuid.uuid4())})

expect_rejection(
    "a patient born in the future",
    "INSERT INTO pasien (pasien_id, nama, tanggal_lahir, kelamin) "
    "VALUES (:p, 'Test', '2099-01-01', 'L')",
    {"p": str(uuid.uuid4())})

  rejected  a visit for a patient who does not exist
            insert or update on table "kunjungan" violates foreign key constraint "kunjungan_pasien_id_fkey"
  rejected  blood pressure with diastolic above systolic
            new row for relation "pengkajian" violates check constraint "pengkajian_check"
  rejected  a patient with an invalid sex code
            new row for relation "pasien" violates check constraint "pasien_kelamin_check"
  rejected  a patient born in the future
            new row for relation "pasien" violates check constraint "pasien_tanggal_lahir_check"


## Step 7 — Load and build the fact table

The ETL lives in `build_warehouse.py`, so it is reused here rather than reimplemented. It
migrates the 21 source assessments and generates 60 further synthetic patients, under a
fixed seed so the build is reproducible.

In [14]:
import build_warehouse as bw

conn = bw.connect()
conn.autocommit = False
cur = conn.cursor()

wilayah_ids = list(bw.lookup(cur, "dim_wilayah", "kota", "wilayah_id").values())
dokter    = bw.lookup(cur, "dim_dokter", "nama", "dokter_id")
diagnosis = bw.lookup(cur, "dim_diagnosis", "nama", "diagnosis_id")
tes       = bw.lookup(cur, "dim_jenis_tes", "nama", "jenis_tes_id")
obat      = bw.lookup(cur, "dim_obat", "nama", "obat_id")

sp, sv = bw.migrate_source(cur, wilayah_ids, dokter, diagnosis, tes, obat)
gp, gv = bw.generate(cur, 60, wilayah_ids, dokter, diagnosis, tes, obat)
bw.run_sql_file(cur, "03_build_fact.sql")
conn.commit()

print(f"source   : {sp} patient, {sv} visits")
print(f"generated: {gp} patients, {gv} visits")

source   : 1 patient, 21 visits
generated: 60 patients, 146 visits


In [15]:
pd.read_sql("""
SELECT relname AS table, n_live_tup AS rows
FROM pg_stat_user_tables ORDER BY relname
""", engine)

,table,rows
0,biodata_pasien,1
1,dim_diagnosis,0
2,dim_dokter,0
3,dim_jenis_tes,0
4,dim_obat,0
5,dim_wilayah,0
6,fact_checkup,0
7,farmasi,21
8,hasil_lab,0
9,konsultasi,0


### Verify the grain and the integrity

One fact row per visit, and no orphans — the two properties the original schema could not
achieve.

In [16]:
checks = pd.read_sql("""
SELECT
  (SELECT COUNT(*) FROM kunjungan)                          AS visits,
  (SELECT COUNT(*) FROM fact_checkup)                       AS fact_rows,
  (SELECT COUNT(*) FROM hasil_lab l
     LEFT JOIN kunjungan k USING (kunjungan_id)
    WHERE k.kunjungan_id IS NULL)                           AS orphan_labs,
  (SELECT COUNT(*) FROM fact_checkup WHERE bmi <= 0 OR bmi > 60) AS bad_bmi
""", engine)
display(checks)

row = checks.iloc[0]
assert row.fact_rows == row.visits, "fact grain is not one row per visit"
assert row.orphan_labs == 0, "orphaned lab rows present"
assert row.bad_bmi == 0, "implausible BMI values"
print("one fact row per visit, zero orphans, BMI all plausible")

,visits,fact_rows,orphan_labs,bad_bmi
0,167,167,0,0


one fact row per visit, zero orphans, BMI all plausible


## Step 8 — Query the warehouse

In [17]:
pd.read_sql("SELECT * FROM v_nutrition_status", engine)

,bmi_kategori,kunjungan,pasien,bmi_rata2,umur_rata2_tahun
0,Gizi Kurang,14,12,13.26,5.7
1,Normal,108,55,16.07,5.7
2,Gizi Lebih,26,22,18.87,6.9
3,Obesitas,19,2,30.48,6.0


In [18]:
pd.read_sql("SELECT * FROM v_diagnosis_frequency LIMIT 6", engine)

,diagnosis,kode_icd10,kunjungan,persen
0,Alergi,T78.4,33,19.8
1,Batuk,R05,33,19.8
2,Gizi Kurang,E44.0,29,17.4
3,Sehat,Z00.1,27,16.2
4,Flu,J00,24,14.4
5,Demam,R50.9,21,12.6


### The query the old schema made impossible

Lab results outside their reference range, per patient. It needs the `pasien_id` foreign key
that the original `CREATE TABLE` omitted, and the reference ranges that `dim_jenis_tes` now
holds.

In [19]:
abnormal = pd.read_sql("SELECT * FROM v_abnormal_labs LIMIT 8", engine)
total = pd.read_sql("SELECT COUNT(*) AS n FROM v_abnormal_labs", engine).n.iloc[0]
print(f"{total} results outside their reference range")
abnormal

137 results outside their reference range


,pasien,tes,nilai,satuan,ref_bawah,ref_atas,arah,tanggal
0,Wulan Pratama,Hemoglobin,107.02,g/dL,11.5,15.5,Tinggi,2026-11-15
1,Galih Permana,Hemoglobin,92.70,g/dL,11.5,15.5,Tinggi,2026-11-10
2,Wulan Saputra,Gula Darah,15.70,mg/dL,70.0,140.0,Rendah,2026-11-09
3,Wulan Saputra,Hemoglobin,164.84,g/dL,11.5,15.5,Tinggi,2026-11-09
4,Fajar Rahayu,Gula Darah,68.24,mg/dL,70.0,140.0,Rendah,2026-10-24
5,Citra Santoso,Hemoglobin,42.55,g/dL,11.5,15.5,Tinggi,2026-10-14
6,Wulan Santoso,Hemoglobin,158.09,g/dL,11.5,15.5,Tinggi,2026-10-13
7,Citra Maulana,Hemoglobin,142.84,g/dL,11.5,15.5,Tinggi,2026-10-04


In [20]:
cur.close()
conn.close()
print("done")

done


---

## Summary

| | |
|---|---|
| Source | 5 flat tables, no keys between three of them, `fact_checkup` at 0 rows |
| 1NF | blood pressure split into systolic/diastolic; address decomposed; stored age replaced by birth date |
| 2NF | parental height moved from every checkup row to one row per patient |
| 3NF | doctor, diagnosis, test type, drug and region extracted to dimensions |
| Integrity | `pasien_id` restored; a `kunjungan` (visit) entity makes every join path unambiguous |
| Grain | one fact row per visit, with labs and prescriptions aggregated first — avoiding a 194,481-row fan trap |
| Enforced | primary keys, foreign keys and check constraints, demonstrated by four rejected inserts |

### A note on the data

All of it is synthetic. The single patient in the source spreadsheets was already dummy
data — named "John Doe", UUID identifiers, one doctor called "Dr. Andi", generic
diagnoses — and the 60 additional patients are generated from fixed name lists under a
seeded RNG. No real patient information is involved.

The source records are also internally inconsistent: height plateaus at 90 cm while weight
reaches 27.5 kg over ten years, so that patient's BMI climbs to 34 and classifies as
obese. The arithmetic is right; the input never followed a growth curve. The project README
covers this and the other caveats.